In [212]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [213]:
# read in the data
euro_df = pd.read_csv("European_Food_Waste_Data.csv")
char_df = pd.read_csv("Food_Waste_Characterization_perc.csv")

In [214]:
euro_df

,Type,Denmark_p,Germany_p,Italy_p,Denmark_m,Germany_m,Italy_m
0,Meat,0.00,0.00,0.00,20.13,186.69,143.55
1,Fish,136.50,58.06,66.46,27.89,80.37,37.75
2,Dairy,14.90,89.16,33.45,43.63,218.16,139.81
3,Eggs,1.41,14.42,14.76,1.35,16.90,13.59
4,Cereals,2.06,45.06,18.99,16.42,212.97,44.71
5,Fruits,2.02,99.25,811.30,7.81,135.46,679.84
6,Vegetables,26.61,332.97,868.17,24.05,270.29,570.32
7,Potatoes,0.00,0.00,0.00,9.45,192.46,10.97
8,Sugarbeets,0.00,0.00,0.00,0.00,0.00,0.00
9,Oil crops,1.24,15.18,0.69,0.00,0.01,59.23


In [215]:
char_df

,num,G.num,Food.Wastes,Food.Wastes.Clean,Group,pH,TS.Perc,VS.Perc,TOC,TKN,Fat.Perc,Protein.Perc,Carbohydrate.Perc,TP,TK,C.N.Mixture,BMP
0,1,DP.1,Cheese,Cheese,DP,5.93,0.4986,0.4263,29.80,2.90,0.2320,0.1850,0.5830,1.14,0.17,17.38,561.0
1,2,DP.2,Milk,Milk,DP,6.76,0.1096,0.1056,5.39,1.66,0.1560,0.3340,0.5100,1.70,1.10,3.32,231.0
2,3,DP.3,Baby milk,Baby_milk,DP,7.15,0.1138,0.1089,4.41,1.95,0.2380,0.1035,0.6585,1.06,1.69,17.30,315.0
3,4,DP.4,Yogurt/yogurt drink,Yogurt_yogurt_drink,DP,4.30,0.3090,0.3030,12.80,0.91,0.0510,0.1430,0.8060,0.70,0.14,14.06,450.0
4,5,DP.5,Cottage cheese,Cottage_cheese,DP,6.37,0.3956,0.3727,51.00,1.54,0.1950,0.0450,0.7600,1.09,0.60,17.31,591.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,84,OT.84,Rice,Rice,OT,6.13,0.8988,0.7176,40.50,1.51,0.0100,0.0800,0.9100,0.16,0.11,17.99,463.0
84,85,OT.85,Cafeteria wastes (CW),Cafeteria_wastes_CW,OT,6.22,0.1425,0.1350,48.95,1.96,0.0400,0.1700,0.7900,0.17,0.89,24.90,370.0
85,86,OT.86,Tea,Tea,OT,4.92,0.9262,0.6360,48.20,3.74,0.0001,0.2340,0.7659,0.39,2.05,12.89,235.0
86,87,OT.87,Spent coffee grounds,Spent_coffee_grounds,OT,6.13,0.3801,0.3221,46.40,1.95,0.6012,0.3988,0.0000,0.05,0.45,18.22,378.0


## Cleaning
Mainly just removing Sugarbeets because it's a row of all 0s so it doesn't contribute anything to the analysis

In [216]:
euro_df.drop(euro_df.loc[euro_df['Type'] == 'Sugarbeets'].index, inplace=True)

## Processing

We have the mass of each category of food waste from the euro data (in kt). We want to convert this to the percentage of protein, carbs, and fat comprising the volatile solids in this mass. Food waste inputs are assumed to be pure volatile solids. To this end, we will calculate
1. Euro food waste data is in kt, so first step is dividing by total mass of the stream to get each food group's % contribution
2. this mass % is multiplied by the VS % of each food group from the characterization data, then normalized so they sum to 1, to get the percentage of total VS accounted for by each of the food groups
3. each % per food group is then multiplied by the fat/protein/carb % of said food group, and those are summed across fat/protein/carb to get the % of fat/protein/carbs per unit mass of total VS in each food waste stream
4. last step is multiplying those last %s by the mass per feed step to get the values that go into the feed file

In [217]:
# extract total mass of each country's waste streams
denmark_processing_total = euro_df.loc[euro_df['Type'] == 'Total', 'Denmark_p'].values[0]
denmark_manufacturing_total = euro_df.loc[euro_df['Type'] == 'Total', 'Denmark_m'].values[0]

germany_processing_total = euro_df.loc[euro_df['Type'] == 'Total', 'Germany_p'].values[0]
germany_manufacturing_total = euro_df.loc[euro_df['Type'] == 'Total', 'Germany_m'].values[0]

italy_processing_total = euro_df.loc[euro_df['Type'] == 'Total', 'Italy_p'].values[0]
italy_manufacturing_total = euro_df.loc[euro_df['Type'] == 'Total', 'Italy_m'].values[0]

# create new series with percentages of each food type
euro_df['Denmark_p_perc'] = euro_df['Denmark_p']/denmark_processing_total
euro_df['Denmark_m_perc'] = euro_df['Denmark_m']/denmark_manufacturing_total

euro_df['Germany_p_perc'] = euro_df['Germany_p']/germany_processing_total
euro_df['Germany_m_perc'] = euro_df['Germany_m']/germany_manufacturing_total

euro_df['Italy_p_perc'] = euro_df['Italy_p']/italy_processing_total
euro_df['Italy_m_perc'] = euro_df['Italy_m']/italy_manufacturing_total

euro_df[['Type', 'Denmark_p_perc', 'Denmark_m_perc', 'Germany_p_perc', 'Germany_m_perc', 'Italy_p_perc', 'Italy_m_perc']]

,Type,Denmark_p_perc,Denmark_m_perc,Germany_p_perc,Germany_m_perc,Italy_p_perc,Italy_m_perc
0,Meat,0.000000,0.135346,0.000000,0.142174,0.000000,0.084452
1,Fish,0.738876,0.187521,0.088763,0.061206,0.036641,0.022209
2,Dairy,0.080654,0.293350,0.136309,0.166140,0.018442,0.082252
3,Eggs,0.007632,0.009077,0.022046,0.012870,0.008138,0.007995
4,Cereals,0.011151,0.110401,0.068889,0.162187,0.010470,0.026303
5,Fruits,0.010934,0.052511,0.151735,0.103160,0.447291,0.399958
6,Vegetables,0.144040,0.161702,0.509051,0.205840,0.478644,0.335526
7,Potatoes,0.000000,0.063538,0.000000,0.146568,0.000000,0.006454
9,Oil crops,0.006712,0.000000,0.023207,0.000008,0.000380,0.034846
10,Total,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [218]:
# get VS% and fat/carb/prot mix for each food group used in euro df

# set up new dataframe to hold calculated percentages
euro_char_df = pd.DataFrame()
euro_char_df['Type'] = euro_df['Type'].drop(euro_df.index[-1])
euro_char_df[['VS_Perc', 'Fat_Perc', 'Protein_Perc', 'Carbohydrate_Perc']] = 0.0

# function to populate the row at food_type with the average values from food_df
def avg_to_row(food_type, food_df:pd.DataFrame):
    euro_char_df.loc[euro_char_df['Type'] == food_type, 'VS_Perc'] = food_df['VS.Perc'].mean()
    euro_char_df.loc[euro_char_df['Type'] == food_type, 'Fat_Perc'] = food_df['Fat.Perc'].mean()
    euro_char_df.loc[euro_char_df['Type'] == food_type, 'Protein_Perc'] = food_df['Protein.Perc'].mean()
    euro_char_df.loc[euro_char_df['Type'] == food_type, 'Carbohydrate_Perc'] = food_df['Carbohydrate.Perc'].mean()

# populate each row with the appropriate item(s) from food characterization df
avg_to_row('Meat', char_df.loc[char_df['Group'] == 'MP'])
avg_to_row('Fish', char_df.loc[char_df['Group'] == 'FP'])
avg_to_row('Dairy', char_df.loc[char_df['Group'] == 'DP'])
avg_to_row('Eggs', char_df.loc[char_df['Group'] == 'EP'])
avg_to_row('Cereals', char_df.loc[char_df['Group'] == 'CP'])
avg_to_row('Fruits', char_df.loc[char_df['Food.Wastes.Clean'] == 'Mix_fruit_waste'])
avg_to_row('Vegetables', char_df.loc[char_df['Food.Wastes.Clean'] == 'Mixed_vegetable_wastes'])
avg_to_row('Potatoes', char_df.loc[char_df['Food.Wastes.Clean'] == 'Potatoes'])
avg_to_row('Oil crops', char_df.loc[char_df['Group'] == 'FOG'])

euro_char_df

,Type,VS_Perc,Fat_Perc,Protein_Perc,Carbohydrate_Perc
0,Meat,0.439075,0.577375,0.251663,0.172212
1,Fish,0.392520,0.655320,0.274840,0.069780
2,Dairy,0.366450,0.284333,0.140533,0.575133
3,Eggs,0.227725,0.729875,0.189975,0.079375
4,Cereals,0.760100,0.038267,0.111900,0.849833
5,Fruits,0.180400,0.002100,0.031000,0.966900
6,Vegetables,0.223400,0.008700,0.153000,0.838300
7,Potatoes,0.105500,0.008000,0.105000,0.779000
9,Oil crops,0.673517,0.999500,0.000500,0.000000


In [219]:
euro_df[['Type', 'Denmark_p_perc', 'Denmark_m_perc', 'Germany_p_perc', 'Germany_m_perc', 'Italy_p_perc', 'Italy_m_perc']]

,Type,Denmark_p_perc,Denmark_m_perc,Germany_p_perc,Germany_m_perc,Italy_p_perc,Italy_m_perc
0,Meat,0.000000,0.135346,0.000000,0.142174,0.000000,0.084452
1,Fish,0.738876,0.187521,0.088763,0.061206,0.036641,0.022209
2,Dairy,0.080654,0.293350,0.136309,0.166140,0.018442,0.082252
3,Eggs,0.007632,0.009077,0.022046,0.012870,0.008138,0.007995
4,Cereals,0.011151,0.110401,0.068889,0.162187,0.010470,0.026303
5,Fruits,0.010934,0.052511,0.151735,0.103160,0.447291,0.399958
6,Vegetables,0.144040,0.161702,0.509051,0.205840,0.478644,0.335526
7,Potatoes,0.000000,0.063538,0.000000,0.146568,0.000000,0.006454
9,Oil crops,0.006712,0.000000,0.023207,0.000008,0.000380,0.034846
10,Total,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [220]:
# create new columns with the % of total VS accounted for by each food group
euro_char_VS_perc = euro_char_df['VS_Perc'].to_numpy()

def build_vs_perc(stream_name):
    stream_vs_percs = euro_df[stream_name+'_perc'].to_numpy()[:-1] * euro_char_VS_perc
    stream_vs_percs = stream_vs_percs/np.sum(stream_vs_percs)
    euro_df[stream_name+'_vs_perc'] = np.concatenate([stream_vs_percs, [np.sum(stream_vs_percs)]])

build_vs_perc('Denmark_p')
build_vs_perc('Denmark_m')
build_vs_perc('Germany_p')
build_vs_perc('Germany_m')
build_vs_perc('Italy_p')
build_vs_perc('Italy_m')

euro_df[['Type', 'Denmark_p_vs_perc', 'Denmark_m_vs_perc', 'Germany_p_vs_perc', 'Germany_m_vs_perc', 'Italy_p_vs_perc', 'Italy_m_vs_perc']]

,Type,Denmark_p_vs_perc,Denmark_m_vs_perc,Germany_p_vs_perc,Germany_m_vs_perc,Italy_p_vs_perc,Italy_m_vs_perc
0,Meat,0.000000,0.156876,0.000000,0.176539,0.000000,0.137841
1,Fish,0.787113,0.194305,0.116565,0.067942,0.065724,0.032405
2,Dairy,0.080213,0.283775,0.167115,0.172176,0.030883,0.112044
3,Eggs,0.004717,0.005457,0.016796,0.008289,0.008468,0.006768
4,Cereals,0.023003,0.221523,0.175183,0.348635,0.036366,0.074321
5,Fruits,0.005353,0.025007,0.091579,0.052630,0.368742,0.268212
6,Vegetables,0.087331,0.095361,0.380468,0.130045,0.488645,0.278636
7,Potatoes,0.000000,0.017695,0.000000,0.043730,0.000000,0.002531
9,Oil crops,0.012269,0.000000,0.052294,0.000015,0.001171,0.087242
10,Total,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
# build a new table with each stream as a row, with fat/protein/carbs in the columns
euro_char_fat_perc = euro_char_df[['Fat_Perc']].to_numpy()
euro_char_protein_perc = euro_char_df[['Protein_Perc']].to_numpy()
euro_char_carb_perc = euro_char_df[['Carbohydrate_Perc']].to_numpy()

euro_macros_df = pd.DataFrame(columns=['stream', 'Fat.Perc', 'Protein.Perc', 'Carbohydrate.Perc'])

def build_char_row(stream_name, full_name):
    stream_vs_percs = euro_df[[stream_name+'_vs_perc']].to_numpy()[:-1]
    stream_p_fat_perc = np.sum(stream_vs_percs * euro_char_fat_perc)
    stream_p_protein_perc = np.sum(stream_vs_percs * euro_char_protein_perc)
    stream_p_carb_perc = np.sum(stream_vs_percs * euro_char_carb_perc)
    total_perc = stream_p_fat_perc + stream_p_protein_perc + stream_p_carb_perc
    euro_macros_df.loc[len(euro_macros_df)] = [full_name, stream_p_fat_perc/total_perc,
                                               stream_p_protein_perc/total_perc, stream_p_carb_perc/total_perc]

build_char_row('Denmark_p', 'Denmark_production')
build_char_row('Denmark_m', 'Denmark_manufacturing')
build_char_row('Germany_p', 'Germany_production')
build_char_row('Germany_m', 'Germany_manufacturing')
build_char_row('Italy_p', 'Italy_production')
build_char_row('Italy_m', 'Italy_manufacturing')

euro_macros_df

,stream,Fat.Perc,Protein.Perc,Carbohydrate.Perc
0,Denmark_production,0.556004,0.244619,0.199377
1,Denmark_manufacturing,0.312620,0.176116,0.511264
2,Germany_production,0.198640,0.139395,0.661964
3,Germany_manufacturing,0.217386,0.154703,0.627910
4,Italy_production,0.065621,0.114277,0.820102
5,Italy_manufacturing,0.230694,0.120212,0.649094


In [226]:
std_columns = ["time", "q_in", "S_ch4", "S_IC", "S_IN", "S_h2o", "X_ch", "X_pr", "X_li", "X_bac", "S_gas_ch4", "S_gas_co2"]

cont_length = 100
cont_times = np.arange(cont_length)
cont_quants = np.repeat(5, cont_length)
cont_water = np.repeat(960.51175, cont_length)

batch_length = 10
batch_times = np.arange(0, 100, 10)
batch_quants = np.repeat(100, batch_length)
batch_water = np.repeat(960.51175, batch_length)

# assemble a continuous and batch version of inputs for each combination
for index, row in euro_macros_df.iterrows():
    # Batch
    batch_df = pd.DataFrame(0, index=np.arange(batch_length), columns=std_columns)
    batch_df['time'] = batch_times
    batch_df['q_in'] = batch_quants
    batch_df['S_h2o'] = batch_water
    batch_df['X_ch'] = np.repeat(round(row['Carbohydrate.Perc']*10, 5), batch_length)
    batch_df['X_pr'] = np.repeat(round(row['Protein.Perc']*10, 5), batch_length)
    batch_df['X_li'] = np.repeat(round(row['Fat.Perc']*10, 5), batch_length)
    batch_df.to_csv(f'Countries/Event/{row['stream']}_event.csv', index=False)

for index, row in euro_macros_df.iterrows():
    # Continuous
    cont_df = pd.DataFrame(0, index=np.arange(cont_length), columns=std_columns)
    cont_df['time'] = cont_times
    cont_df['q_in'] = cont_quants
    cont_df['S_h2o'] = cont_water
    cont_df['X_ch'] = np.repeat(round(row['Carbohydrate.Perc']*74, 5), cont_length)
    cont_df['X_pr'] = np.repeat(round(row['Protein.Perc']*74, 5), cont_length)
    cont_df['X_li'] = np.repeat(round(row['Fat.Perc']*74, 5), cont_length)
    cont_df.to_csv(f'Countries/Continuous/{row['stream']}_continuous.csv', index=False)